## Deep Compression: Aggressive Pruning Experiment

Extends the deep compression pipeline with a higher pruning threshold (s=1.1 std) to push sparsity to ~80%, compared to the ~63% achieved in the baseline run.

### Pipeline

1. **Pruning**: Apply std-based pruning at s=1.1 for ~80% sparsity, then fine-tune to recover accuracy.
2. **Quantization**: Apply 5-bit k-means quantization on the pruned weights.
3. **Huffman Coding**: Compute entropy-coded bit-width and final compression ratio.

**Model**: VGG16_half | **Dataset**: CIFAR-10 | **Result**: ~40x compression at 90.84% accuracy

In [ ]:
from vgg16 import VGG16, VGG16_half
from train_util import train, finetune_after_prune, test
from quantize import quantize_whole_model
from huffman_coding import huffman_coding
from summary import summary
import torch
import numpy as np
from prune import prune

device = 'cuda' if torch.cuda.is_available() else 'cpu'

/content/summary.py:10: SyntaxWarning: invalid escape sequence '\%'
  print("Layer id\tType\t\tParameter\tNon-zero parameter\tSparsity(\%)")


### Full-precision model training

In [ ]:
net = VGG16_half()
net = net.to(device)

# Uncomment to load pretrained weights
net.load_state_dict(torch.load("net_before_pruning.pt"))

# I loaded the pretrained weights since training has already been completed
# train(net, epochs=150, batch_size=128, lr=0.1, reg=5e-4)

<All keys matched successfully>

In [ ]:
# Load the best weight paramters
net.load_state_dict(torch.load("net_before_pruning.pt"))
test(net)

Test Loss=0.3284, Test accuracy=0.9230


0.923

In [ ]:
print("-----Summary before pruning-----")
summary(net)
print("-------------------------------")

-----Summary before pruning-----
Layer id	Type		Parameter	Non-zero parameter	Sparsity(\%)
1		Convolutional	864		864			0.000000
2		BatchNorm	N/A		N/A			N/A
3		ReLU		N/A		N/A			N/A
4		Convolutional	9216		9216			0.000000
5		BatchNorm	N/A		N/A			N/A
6		ReLU		N/A		N/A			N/A
7		Convolutional	18432		18432			0.000000
8		BatchNorm	N/A		N/A			N/A
9		ReLU		N/A		N/A			N/A
10		Convolutional	36864		36864			0.000000
11		BatchNorm	N/A		N/A			N/A
12		ReLU		N/A		N/A			N/A
13		Convolutional	73728		73728			0.000000
14		BatchNorm	N/A		N/A			N/A
15		ReLU		N/A		N/A			N/A
16		Convolutional	147456		147456			0.000000
17		BatchNorm	N/A		N/A			N/A
18		ReLU		N/A		N/A			N/A
19		Convolutional	147456		147456			0.000000
20		BatchNorm	N/A		N/A			N/A
21		ReLU		N/A		N/A			N/A
22		Convolutional	294912		294912			0.000000
23		BatchNorm	N/A		N/A			N/A
24		ReLU		N/A		N/A			N/A
25		Convolutional	589824		589824			0.000000
26		BatchNorm	N/A		N/A			N/A
27		ReLU		N/A		N/A			N/A
28		Convolutional	589824		589824			0.000000
29		Batch

### Pruning & Finetune with pruned connections

#### Pruning Configuration

In [ ]:
net.load_state_dict(torch.load("net_before_pruning.pt"))

<All keys matched successfully>

In [ ]:
# Test accuracy before fine-tuning
prune(net, method='std', q=45.0, s=1.1)
test(net)

Test Loss=3.3254, Test accuracy=0.2312


0.2312

In [ ]:
# Check sparsity per layer
for n, m in net.named_modules():
    if hasattr(m, 'sparsity'):
        print(f"Layer {n} | Sparsity: {m.sparsity:.4f}")

Layer features.0 | Sparsity: 0.8218
Layer features.3 | Sparsity: 0.8278
Layer features.7 | Sparsity: 0.7623
Layer features.10 | Sparsity: 0.7377
Layer features.14 | Sparsity: 0.7404
Layer features.17 | Sparsity: 0.7336
Layer features.20 | Sparsity: 0.7470
Layer features.24 | Sparsity: 0.7650
Layer features.27 | Sparsity: 0.7932
Layer features.30 | Sparsity: 0.8344
Layer features.34 | Sparsity: 0.8463
Layer features.37 | Sparsity: 0.8665
Layer features.40 | Sparsity: 0.9085
Layer classifer.0 | Sparsity: 0.8433
Layer classifer.3 | Sparsity: 0.8349
Layer classifer.6 | Sparsity: 0.8352


In [ ]:
layer_sparsities = [m.sparsity for m in net.modules() if hasattr(m, 'sparsity')]
avg_sparsity = sum(layer_sparsities) / len(layer_sparsities)
print(f"Model Sparsity: {avg_sparsity:.4f}")

Model Sparsity: 0.8061


In [ ]:
# Uncomment to load pretrained weights
net.load_state_dict(torch.load("net_after_pruning.pt"))

# I loaded the pretrained weights since fine-tuned training has already been completed

# Comment if you have loaded pretrained weights
# finetune_after_prune(net, epochs=60, batch_size=128, lr=0.01, reg=1e-4)

<All keys matched successfully>

In [ ]:
# Load the best weight paramters
net.load_state_dict(torch.load("net_after_pruning.pt"))
test(net)

Test Loss=0.3510, Test accuracy=0.9094


0.9094

In [ ]:
print("-----Summary After pruning-----")
summary(net)
print("-------------------------------")

-----Summary After pruning-----
Layer id	Type		Parameter	Non-zero parameter	Sparsity(\%)
1		Convolutional	864		154			0.821759
2		BatchNorm	N/A		N/A			N/A
3		ReLU		N/A		N/A			N/A
4		Convolutional	9216		1587			0.827799
5		BatchNorm	N/A		N/A			N/A
6		ReLU		N/A		N/A			N/A
7		Convolutional	18432		4381			0.762316
8		BatchNorm	N/A		N/A			N/A
9		ReLU		N/A		N/A			N/A
10		Convolutional	36864		9671			0.737657
11		BatchNorm	N/A		N/A			N/A
12		ReLU		N/A		N/A			N/A
13		Convolutional	73728		19143			0.740356
14		BatchNorm	N/A		N/A			N/A
15		ReLU		N/A		N/A			N/A
16		Convolutional	147456		39278			0.733629
17		BatchNorm	N/A		N/A			N/A
18		ReLU		N/A		N/A			N/A
19		Convolutional	147456		37313			0.746955
20		BatchNorm	N/A		N/A			N/A
21		ReLU		N/A		N/A			N/A
22		Convolutional	294912		69317			0.764957
23		BatchNorm	N/A		N/A			N/A
24		ReLU		N/A		N/A			N/A
25		Convolutional	589824		121953			0.793238
26		BatchNorm	N/A		N/A			N/A
27		ReLU		N/A		N/A			N/A
28		Convolutional	589824		97697			0.834362
29		BatchNorm	N/

### Quantization

In [ ]:
centers = quantize_whole_model(net, bits=5)
np.save("codebook_vgg16.npy", centers)

Complete 1 layers quantization...
Complete 2 layers quantization...
Complete 3 layers quantization...
Complete 4 layers quantization...
Complete 5 layers quantization...
Complete 6 layers quantization...
Complete 7 layers quantization...
Complete 8 layers quantization...
Complete 9 layers quantization...
Complete 10 layers quantization...
Complete 11 layers quantization...
Complete 12 layers quantization...
Complete 13 layers quantization...
Complete 14 layers quantization...
Complete 15 layers quantization...
Complete 16 layers quantization...


In [ ]:
test(net)

Test Loss=0.3535, Test accuracy=0.9084


0.9084

### Huffman Coding

In [35]:
frequency_map, encoding_map = huffman_coding(net, centers)
np.save("huffman_encoding", encoding_map)
np.save("huffman_freq", frequency_map)

Original storage for each parameter: 5.0000 bits
Average storage for each parameter after Huffman Coding: 4.8117 bits
Complete 1 layers for Huffman Coding...
Original storage for each parameter: 5.0000 bits
Average storage for each parameter after Huffman Coding: 4.7183 bits
Complete 2 layers for Huffman Coding...
Original storage for each parameter: 5.0000 bits
Average storage for each parameter after Huffman Coding: 4.6649 bits
Complete 3 layers for Huffman Coding...
Original storage for each parameter: 5.0000 bits
Average storage for each parameter after Huffman Coding: 4.8318 bits
Complete 4 layers for Huffman Coding...
Original storage for each parameter: 5.0000 bits
Average storage for each parameter after Huffman Coding: 4.7475 bits
Complete 5 layers for Huffman Coding...
Original storage for each parameter: 5.0000 bits
Average storage for each parameter after Huffman Coding: 4.7985 bits
Complete 6 layers for Huffman Coding...
Original storage for each parameter: 5.0000 bits
Ave

In [36]:
test(net)

Test Loss=0.3535, Test accuracy=0.9084


0.9084

In [37]:
# Data from Huffman run
huffman_bits = [
    4.8117, 4.7183, 4.6649, 4.8318, 4.7475, 4.7985, 4.8388, 4.8031,
    4.7059, 4.7268, 4.5319, 4.4436, 4.1004, 4.3625, 4.5625, 4.6848
]
# Original Length
original_bits = 5.0

huffman_length = np.average(huffman_bits)

huff_ratio = huffman_length/original_bits
print(huff_ratio)

0.9291625


In [38]:
Ratio = (645398/3811680) * (5/32) * huff_ratio
print(Ratio)

Compression_Rate = 1 / Ratio
print(f"Compression Ratio: {Compression_Rate}")
print(f"Final Test Accuracy: {test(net)}")

0.02458232078665936
Compression Ratio: 40.679641628576114
Test Loss=0.3535, Test accuracy=0.9084
Final Test Accuracy: 0.9084
